In [0]:
dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema","artemzharkov10_silver")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SILVER_CATALOG}.{SILVER_SCHEMA}.checkpoint")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_CATALOG}.{SILVER_SCHEMA}.stream_bitcoin_silver (
        TradeDate TIMESTAMP,
        Open DOUBLE,
        High DOUBLE,
        Low DOUBLE,
        Close DOUBLE,
        Volume DOUBLE
    )
""")

try: 
    spark.sql(f"""
              ALTER TABLE {SILVER_CATALOG}.{SILVER_SCHEMA}.stream_bitcoin_silver 
              ADD CONSTRAINT valid_close_price CHECK (Close > 0)
              """)
    spark.sql(f"""
              ALTER TABLE {SILVER_CATALOG}.{SILVER_SCHEMA}.stream_bitcoin_silver 
              ADD CONSTRAINT valid_trade_date CHECK (TradeDate Is NOT NULL) 
              """)
    spark.sql(f"""
              ALTER TABLE {SILVER_CATALOG}.{SILVER_SCHEMA}.stream_bitcoin_silver 
              ADD CONSTRAINT valid_volume CHECK (Volume > 0)
              """)
except Exception as e:
    print(f"Erore during table verification: {e}")
    